# 6. Agents (ReAct-style, tool-using loops)

An agent is a loop: the model reasons about what to do, optionally calls a tool,
observes the result, and repeats until it has enough information to answer —
contrast with notebook 3's tool-calling, which does exactly one round.
`langgraph.prebuilt.create_react_agent` builds this loop for you; attaching a
`checkpointer` additionally gives the agent multi-turn memory keyed by a `thread_id`.

**A real lesson worth knowing** (found and fixed while building this project —
see `docs/langchain/06-react-agents.md`'s Gotchas): a smaller model can re-call a
tool with *identical* arguments several times after already getting the answer,
if there's no explicit instruction telling it to stop once it has a result. The
`AGENT_SYSTEM_PROMPT` below exists specifically for that.

**Prerequisites:** Ollama running locally with `llama3.2` pulled.

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import create_react_agent

from models.chat_models.ollama_models import SupportedModel, get_chat_model
from tools.datetime_tools import current_date, days_between
from tools.math_tools import adder, divider, multiplier, subtractor

AGENT_TOOLS = [adder, subtractor, multiplier, divider, current_date, days_between]

AGENT_SYSTEM_PROMPT = (
    "You are a helpful assistant with access to tools. Once a tool call has "
    "returned a result that answers the user's question, respond directly "
    "with the final answer in plain text. Never call the same tool with the "
    "same arguments more than once."
)

llm = get_chat_model(SupportedModel.llama3_2)
agent = create_react_agent(llm, tools=AGENT_TOOLS, checkpointer=InMemorySaver(), prompt=AGENT_SYSTEM_PROMPT)

## A single-tool-call request, with a step trace

In [ ]:
thread = {"configurable": {"thread_id": "notebook-demo"}}
result = agent.invoke(
    {"messages": [("human", "How many days are between 2026-01-01 and 2026-03-15?")]}, config=thread
)

for message in result["messages"]:
    message_type = getattr(message, "type", None)
    if message_type == "ai" and getattr(message, "tool_calls", None):
        for tool_call in message.tool_calls:
            print(f"tool_call: {tool_call['name']}({tool_call['args']})")
    elif message_type == "tool":
        print(f"tool_result: {message.content}")
    elif message_type == "ai" and message.content:
        print(f"ai_message: {message.content}")

## Multi-turn memory — same `thread_id`

In [ ]:
result_2 = agent.invoke({"messages": [("human", "And how many weeks is that, roughly?")]}, config=thread)
print(result_2["messages"][-1].content)

## 🧪 Playground

**1. Multi-step chaining is a documented small-model limitation** — try `"add 15 and 27, then multiply the result by 2"` on a *fresh* `thread_id` and inspect the step trace. Does it pick the right tool for both steps, or does `final_answer` diverge from what `steps` actually shows? (See `docs/langchain/06-react-agents.md`'s Gotchas for the known failure mode.)

In [ ]:
# TODO: try the two-step arithmetic request on a new thread_id and print the full step trace


**2. Remove `AGENT_SYSTEM_PROMPT`** (build a second agent without `prompt=...`) and see if you can reproduce the repeat-call behavior with a harder multi-turn question.

In [ ]:
# TODO: agent_no_prompt = create_react_agent(llm, tools=AGENT_TOOLS, checkpointer=InMemorySaver())


**3. Add `web_search`** to `AGENT_TOOLS` (needs `GOOGLE_API_KEY`/`GOOGLE_CSE_ID` in `.env`) and ask something needing current information.

In [ ]:
# TODO: from tools.search_tools import web_search; rebuild AGENT_TOOLS with it included
